# K562 + TNF-alpha + puromycin tradeSeq fits

This notebook runs the package-backed single-lineage tradeSeq backend used by downstream ChronoPack analyses. Experiment-specific behavior is kept in `TRADESEQ_CONFIG`.


In [1]:
find_parent_with <- function(marker) {
  start <- normalizePath(getwd(), mustWork = TRUE)
  parts <- strsplit(start, .Platform$file.sep, fixed = TRUE)[[1]]
  if (parts[1] == "") parts[1] <- .Platform$file.sep
  candidates <- vapply(seq_along(parts), function(i) {
    if (i == 1 && parts[1] == .Platform$file.sep) return(.Platform$file.sep)
    do.call(file.path, as.list(parts[seq_len(i)]))
  }, character(1))
  candidates <- rev(candidates)
  hit <- candidates[file.exists(file.path(candidates, marker))]
  if (!length(hit)) {
    stop("Could not find parent containing ", marker, " from ", start)
  }
  normalizePath(hit[[1]], mustWork = TRUE)
}

repo_root <- find_parent_with("chronopack/R/tradeseq_helpers.R")
experiment_root <- find_parent_with("intmd")
source(file.path(repo_root, "chronopack", "R", "tradeseq_helpers.R"))
load_chronopack_tradeseq_libraries()
suppressPackageStartupMessages(library(dplyr))


Warning message in RNGkind("Mersenne-Twister", "Inversion", "Rounding"):
"non-uniform 'Rounding' sampler used"


In [2]:
TRADESEQ_CONFIG <- list(
  label = "k562_tnfa_puromycin",
  input_dir = file.path(experiment_root, "intmd", "sc-K562-1e6-to-tradeSeq.fixed"),
  n_points = 9,
  workers = 4,
  screen_nknots = 4,
  final_nknots = 8,
  p_threshold = 0.05,
  outputs = list(
    association = "asso_df.1trajectory.nknots4.csv",
    significant = "sig_df.1trajectory.nknots4.csv",
    sce = "sce.1trajectory.nknots8.rds",
    gamList = "gamList.1trajectory.nknots8.rds"
  ),
  legacy_outputs = list(
    association = character(),
    significant = character(),
    sce = character(),
    gamList = character()
  )
)

TRADESEQ_CONFIG


$label
[1] "k562_tnfa_puromycin"

$input_dir
[1] "/new-stg/home/alex/tmp/ChronoPack/scratch_notebooks/k562-tnfa-puromycin.hg38/intmd/sc-K562-1e6-to-tradeSeq.fixed"

$n_points
[1] 9

$workers
[1] 4

$screen_nknots
[1] 4

$final_nknots
[1] 8

$p_threshold
[1] 0.05

$outputs
$outputs$association
[1] "asso_df.1trajectory.nknots4.csv"

$outputs$significant
[1] "sig_df.1trajectory.nknots4.csv"

$outputs$sce
[1] "sce.1trajectory.nknots8.rds"

$outputs$gamList
[1] "gamList.1trajectory.nknots8.rds"


$legacy_outputs
$legacy_outputs$association
character(0)

$legacy_outputs$significant
character(0)

$legacy_outputs$sce
character(0)

$legacy_outputs$gamList
character(0)

In [4]:
# setwd('../intmd/sc-K562-1e6-to-tradeSeq.fixed/')

# counts <- as.matrix(read.csv(gzfile("counts_genes_x_cells.csv.gz"), row.names = 1, check.names = FALSE))
# storage.mode(counts) <- "integer"

# ct = read.csv(gzfile("cell_times.csv.gz"), stringsAsFactors = FALSE)
# libs_full  = read.csv("cell_libsize_full.csv", stringsAsFactors = FALSE)

# time_vec = setNames(ct$time, ct$cell_id)[ colnames(counts) ]
# lib_full = setNames(libs_full$libsize_full, libs_full$cell_id)[ colnames(counts) ]

# genes_sub = read.csv("subset_genes.csv", stringsAsFactors = FALSE)[,1]
# genes_keep = intersect(genes_sub, rownames(counts))
# counts_sub = counts[genes_keep, , drop = FALSE]

# sf = as.numeric(lib_full)
# sf = sf/mean(sf)

# # pseudotime (single trajectory) rescaled to [0,1]
# pt01 = (time_vec - min(time_vec, na.rm = TRUE)) / diff(range(time_vec, na.rm = TRUE))
# PT   = matrix(pt01, ncol = 1); colnames(PT) <- "lineage1"
# CW = matrix(1, nrow = ncol(counts_sub), ncol = 1); colnames(CW) <- "lineage1" # weights: all ones (single lineage)

# # ---- tradeSeq fit ----
# suppressPackageStartupMessages({
#   library(tradeSeq); library(SingleCellExperiment); library(BiocParallel)
# })

# dim(CW)

# offset_mat = matrix(log(sf), nrow = nrow(counts_sub), ncol = ncol(counts_sub), byrow = TRUE)

In [ ]:
fit_result <- run_tradeseq_single_lineage(
  input_dir = TRADESEQ_CONFIG$input_dir,
  n_points = TRADESEQ_CONFIG$n_points,
  workers = TRADESEQ_CONFIG$workers,
  screen_nknots = TRADESEQ_CONFIG$screen_nknots,
  final_nknots = TRADESEQ_CONFIG$final_nknots,
  p_threshold = TRADESEQ_CONFIG$p_threshold,
  output_association = TRADESEQ_CONFIG$outputs$association,
  output_significant = TRADESEQ_CONFIG$outputs$significant,
  output_sce = TRADESEQ_CONFIG$outputs$sce,
  output_gamlist = TRADESEQ_CONFIG$outputs$gamList,
  association_aliases = TRADESEQ_CONFIG$legacy_outputs$association,
  significant_aliases = TRADESEQ_CONFIG$legacy_outputs$significant,
  sce_aliases = TRADESEQ_CONFIG$legacy_outputs$sce,
  gamlist_aliases = TRADESEQ_CONFIG$legacy_outputs$gamList,
  tag = "",
  verbose = TRUE
)

list(
  label = TRADESEQ_CONFIG$label,
  input_dir = fit_result$input_dir,
  counts_dim = fit_result$counts_dim,
  counts_sub_dim = fit_result$counts_sub_dim,
  n_tested = fit_result$n_tested,
  n_qval_significant = fit_result$n_qval_significant,
  n_pvalue_significant = fit_result$n_pvalue_significant,
  final_counts_dim = fit_result$final_counts_dim,
  association_path = fit_result$association_path,
  significant_path = fit_result$significant_path,
  sce_path = fit_result$sce_path,
  gamlist_path = fit_result$gamlist_path
)
